In [ ]:
# 필요한 라이브러리 임포트
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

import matplotlib.font_manager as fm

# 한글 폰트 설정
try:
    font_path = r'C:\Windows\Fonts\malgun.ttf'
    font_prop = fm.FontProperties(fname=font_path)
    plt.rcParams['font.family'] = font_prop.get_name()
except:
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings('ignore')

# 시각화 스타일 설정
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# 데이터 로드
df = pd.read_csv('final.csv')
df.drop(['matchId','gameDuration'], axis=1, inplace=True)

print(f"전체 데이터 수: {len(df):,}건")
print(f"전체 변수 수: {len(df.columns)}개")
print(f"\n승패 비율:")
print(df['blueWins'].value_counts(normalize=True))
df.head()

In [ ]:
# 변수 그룹 정의 (ftest.ipynb와 동일)
target_vars_ordered = [
    ('성장', 'TotalGold', 'blueTotalGold', 'redTotalGold'),
    ('성장', 'TotalExperience', 'blueTotalExperience', 'redTotalExperience'),
    ('성장', 'TotalMinionsKilled', 'blueTotalMinionsKilled', 'redTotalMinionsKilled'),
    ('성장', 'TotalJungleMinionsKilled', 'blueTotalJungleMinionsKilled', 'redTotalJungleMinionsKilled'),
    ('전투', 'Kills', 'blueKills', 'redKills'),
    ('전투', 'Deaths', 'blueDeaths', 'redDeaths'),
    ('전투', 'Assists', 'blueAssists', 'redAssists'),
    ('전투', 'FirstBlood', 'blueFirstBlood', 'redFirstBlood'),
    ('오브젝트', 'Dragons', 'blueDragons', 'redDragons'),
    ('오브젝트', 'FirstDragon', 'blueFirstDragon', 'redFirstDragon'),
    ('오브젝트', 'Heralds', 'blueHeralds', 'redHeralds'),
    ('오브젝트', 'VoidGrubs', 'blueVoidGrubs', 'redVoidGrubs'),
    ('구조물', 'TowersDestroyed', 'blueTowersDestroyed', 'redTowersDestroyed'),
    ('구조물', 'FirstTowerDestroyed', 'blueFirstTurret', 'redFirstTurret'),
    ('구조물', 'PlatesDestroyed', 'bluePlatesDestroyed', 'redPlatesDestroyed'),
    ('구조물', 'InhibitorsDestroyed', 'blueInhibitorsDestroyed', 'redInhibitorsDestroyed'),
    ('시야', 'WardsPlaced', 'blueWardsPlaced', 'redWardsPlaced'),
    ('시야', 'ControlWardsPlaced', 'blueControlWardsPlaced', 'redControlWardsPlaced'),
    ('시야', 'WardsDestroyed', 'blueWardsDestroyed', 'redWardsDestroyed'),
    ('시야', 'ControlWardsDestroyed', 'blueControlWardsDestroyed', 'redControlWardsDestroyed') 
]

print(f"분석 대상 변수 카테고리: {len(set([v[0] for v in target_vars_ordered]))}개")
print(f"분석 대상 변수 총 개수: {len(target_vars_ordered)}개")

In [ ]:
# 승리/패배 팀별 통계 계산 (ftest.ipynb와 동일)
results = []

for category, name, b_col, r_col in target_vars_ordered:
    if b_col in df.columns:
        # 승리 팀 / 패배 팀 데이터 추출
        winner_vals = np.where(df['blueWins'] == 1, df[b_col], df[r_col])
        loser_vals = np.where(df['blueWins'] == 1, df[r_col], df[b_col])
        
        # 통계 계산
        w_mean = np.mean(winner_vals)
        w_std = np.std(winner_vals)
        w_max = np.max(winner_vals)
        w_min = np.min(winner_vals)
        
        l_mean = np.mean(loser_vals)
        l_std = np.std(loser_vals)
        l_max = np.max(loser_vals)
        l_min = np.min(loser_vals)
        
        # T-test
        t_stat, p_val = stats.ttest_ind(winner_vals, loser_vals, equal_var=False)
        p_val_str = "< 0.001" if p_val < 0.001 else f"{p_val:.4f}"

        results.append({
            '구분': category,
            '변수명': name,
            '승리_평균': w_mean,
            '승리_표준편차': w_std,
            '패배_평균': l_mean,
            '패배_표준편차': l_std,
            'T-value': t_stat,
            'P-value': p_val_str,
            'blue_col': b_col,
            'red_col': r_col
        })

stats_df = pd.DataFrame(results)
print("\n=== 승리 팀 vs 패배 팀 기초 통계 ===\n")
print(stats_df[['구분', '변수명', '승리_평균', '패배_평균', 'T-value']].to_string(index=False))

## 1. 전체 변수 분포 시각화 (카테고리별)

In [ ]:
# 카테고리별로 변수 그룹화
categories = ['성장', '전투', '오브젝트', '구조물', '시야']
category_colors = {
    '성장': '#FF6B6B',
    '전투': '#4ECDC4', 
    '오브젝트': '#FFE66D',
    '구조물': '#95E1D3',
    '시야': '#C7CEEA'
}

for category in categories:
    cat_vars = stats_df[stats_df['구분'] == category]
    n_vars = len(cat_vars)
    
    # Blue 팀 변수만 선택 (중복 방지)
    blue_cols = cat_vars['blue_col'].tolist()
    
    # 서브플롯 생성
    fig, axes = plt.subplots(n_vars, 2, figsize=(16, 4*n_vars))
    if n_vars == 1:
        axes = axes.reshape(1, -1)
    
    fig.suptitle(f'[{category}] 변수 분포 (이상치 처리 전)', fontsize=16, fontweight='bold', y=1.0)
    
    for idx, (_, row) in enumerate(cat_vars.iterrows()):
        var_name = row['변수명']
        blue_col = row['blue_col']
        red_col = row['red_col']
        
        # 승리/패배 데이터 분리
        winner_vals = np.where(df['blueWins'] == 1, df[blue_col], df[red_col])
        loser_vals = np.where(df['blueWins'] == 1, df[red_col], df[blue_col])
        
        # 1) 히스토그램 (승리 vs 패배)
        ax1 = axes[idx, 0]
        ax1.hist(winner_vals, bins=50, alpha=0.6, label='승리 팀', color='dodgerblue', edgecolor='black')
        ax1.hist(loser_vals, bins=50, alpha=0.6, label='패배 팀', color='salmon', edgecolor='black')
        ax1.set_title(f'{var_name} - 승리 vs 패배 분포', fontsize=12, fontweight='bold')
        ax1.set_xlabel('값', fontsize=10)
        ax1.set_ylabel('빈도', fontsize=10)
        ax1.legend()
        ax1.grid(alpha=0.3)
        
        # 2) 박스플롯 (이상치 확인)
        ax2 = axes[idx, 1]
        data_to_plot = [winner_vals, loser_vals]
        bp = ax2.boxplot(data_to_plot, labels=['승리 팀', '패배 팀'], patch_artist=True, 
                         notch=True, showmeans=True)
        bp['boxes'][0].set_facecolor('dodgerblue')
        bp['boxes'][1].set_facecolor('salmon')
        ax2.set_title(f'{var_name} - 박스플롯 (이상치 확인)', fontsize=12, fontweight='bold')
        ax2.set_ylabel('값', fontsize=10)
        ax2.grid(alpha=0.3)
        
        # 통계 정보 추가
        ax2.text(0.02, 0.98, f"T-value: {row['T-value']:.2f}", 
                transform=ax2.transAxes, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    plt.show()
    print(f"\n{'='*80}\n")

## 2. Blue vs Red 팀 직접 비교 (Raw Data)

In [ ]:
# Blue vs Red 팀 원본 데이터 비교
for category in categories:
    cat_vars = stats_df[stats_df['구분'] == category]
    n_vars = len(cat_vars)
    
    fig, axes = plt.subplots(n_vars, 2, figsize=(16, 4*n_vars))
    if n_vars == 1:
        axes = axes.reshape(1, -1)
    
    fig.suptitle(f'[{category}] Blue vs Red 팀 비교', fontsize=16, fontweight='bold', y=1.0)
    
    for idx, (_, row) in enumerate(cat_vars.iterrows()):
        var_name = row['변수명']
        blue_col = row['blue_col']
        red_col = row['red_col']
        
        # 1) 히스토그램 (Blue vs Red)
        ax1 = axes[idx, 0]
        ax1.hist(df[blue_col], bins=50, alpha=0.6, label='Blue 팀', color='blue', edgecolor='black')
        ax1.hist(df[red_col], bins=50, alpha=0.6, label='Red 팀', color='red', edgecolor='black')
        ax1.set_title(f'{var_name} - Blue vs Red 분포', fontsize=12, fontweight='bold')
        ax1.set_xlabel('값', fontsize=10)
        ax1.set_ylabel('빈도', fontsize=10)
        ax1.legend()
        ax1.grid(alpha=0.3)
        
        # 2) 산점도 (Blue vs Red)
        ax2 = axes[idx, 1]
        colors = ['dodgerblue' if w == 1 else 'salmon' for w in df['blueWins']]
        ax2.scatter(df[blue_col], df[red_col], alpha=0.3, c=colors, s=10)
        
        # 대각선 (동일값) 표시
        max_val = max(df[blue_col].max(), df[red_col].max())
        min_val = min(df[blue_col].min(), df[red_col].min())
        ax2.plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=1, alpha=0.5, label='동일값')
        
        ax2.set_title(f'{var_name} - Blue vs Red 상관관계', fontsize=12, fontweight='bold')
        ax2.set_xlabel('Blue 팀', fontsize=10)
        ax2.set_ylabel('Red 팀', fontsize=10)
        ax2.legend(['동일값', 'Blue 승리', 'Red 승리'])
        ax2.grid(alpha=0.3)
        
        # 상관계수 표시
        corr = df[[blue_col, red_col]].corr().iloc[0, 1]
        ax2.text(0.02, 0.98, f"상관계수: {corr:.3f}", 
                transform=ax2.transAxes, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))
    
    plt.tight_layout()
    plt.show()
    print(f"\n{'='*80}\n")

## 3. 이상치 탐지 및 요약 통계

In [ ]:
# 이상치 탐지 (IQR 방법)
outlier_summary = []

for category, name, b_col, r_col in target_vars_ordered:
    if b_col in df.columns:
        for col, team in [(b_col, 'Blue'), (r_col, 'Red')]:
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            
            outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
            n_outliers = len(outliers)
            outlier_pct = (n_outliers / len(df)) * 100
            
            outlier_summary.append({
                '카테고리': category,
                '변수명': name,
                '팀': team,
                'Q1': Q1,
                'Q3': Q3,
                'IQR': IQR,
                '하한': lower_bound,
                '상한': upper_bound,
                '이상치 개수': n_outliers,
                '이상치 비율(%)': outlier_pct
            })

outlier_df = pd.DataFrame(outlier_summary)

print("\n=== 이상치 탐지 요약 (IQR 방법) ===")
print(f"총 변수 수: {len(outlier_df)}개")
print(f"\n이상치가 가장 많은 상위 10개 변수:")
print(outlier_df.nlargest(10, '이상치 비율(%)')[['카테고리', '변수명', '팀', '이상치 개수', '이상치 비율(%)']].to_string(index=False))

In [ ]:
# 이상치 비율 시각화
plt.figure(figsize=(14, 8))

# 카테고리별 평균 이상치 비율
category_outliers = outlier_df.groupby('카테고리')['이상치 비율(%)'].mean().sort_values(ascending=False)

bars = plt.barh(category_outliers.index, category_outliers.values, 
                color=[category_colors.get(c, 'gray') for c in category_outliers.index],
                edgecolor='black', alpha=0.8)

plt.xlabel('평균 이상치 비율 (%)', fontsize=12, fontweight='bold')
plt.title('카테고리별 평균 이상치 비율 (이상치 처리 전)', fontsize=14, fontweight='bold')
plt.grid(axis='x', alpha=0.3)

# 값 표시
for bar in bars:
    width = bar.get_width()
    plt.text(width + 0.1, bar.get_y() + bar.get_height()/2, 
             f'{width:.2f}%', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 4. 변수 간 상관관계 히트맵

In [ ]:
# Blue 팀 변수만 선택하여 상관관계 분석
blue_cols = [row['blue_col'] for _, row in stats_df.iterrows()]
blue_cols_clean = [col.replace('blue', '') for col in blue_cols]

# 상관계수 계산
corr_matrix = df[blue_cols].corr()
corr_matrix.columns = blue_cols_clean
corr_matrix.index = blue_cols_clean

# 히트맵 그리기
plt.figure(figsize=(16, 14))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', 
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('변수 간 상관관계 히트맵 (Blue 팀 기준, 이상치 처리 전)', 
          fontsize=16, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# 상관관계가 높은 변수 쌍 출력
print("\n=== 상관관계가 높은 변수 쌍 (|r| > 0.7) ===")
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        corr_val = corr_matrix.iloc[i, j]
        if abs(corr_val) > 0.7:
            high_corr_pairs.append({
                '변수1': corr_matrix.columns[i],
                '변수2': corr_matrix.columns[j],
                '상관계수': corr_val
            })

if high_corr_pairs:
    high_corr_df = pd.DataFrame(high_corr_pairs).sort_values('상관계수', key=abs, ascending=False)
    print(high_corr_df.to_string(index=False))
else:
    print("상관관계가 0.7 이상인 변수 쌍이 없습니다.")

## 5. 요약 통계 및 데이터 저장

In [ ]:
# 전체 통계 요약
print("\n" + "="*80)
print("전체 데이터 요약 통계 (이상치 처리 전)")
print("="*80)

summary_stats = []
for col in blue_cols:
    col_name = col.replace('blue', '')
    summary_stats.append({
        '변수명': col_name,
        '평균': df[col].mean(),
        '표준편차': df[col].std(),
        '최소값': df[col].min(),
        '25%': df[col].quantile(0.25),
        '중앙값': df[col].median(),
        '75%': df[col].quantile(0.75),
        '최대값': df[col].max(),
        '왜도': df[col].skew(),
        '첨도': df[col].kurtosis()
    })

summary_df = pd.DataFrame(summary_stats)
print(summary_df.to_string(index=False))

# CSV 저장 (선택사항)
# summary_df.to_csv('raw_data_summary_stats.csv', index=False, encoding='utf-8-sig')
# outlier_df.to_csv('raw_data_outlier_analysis.csv', index=False, encoding='utf-8-sig')
# print("\n✅ 통계 데이터가 CSV 파일로 저장되었습니다.")

In [ ]:
# ftest.ipynb의 실전 검증 데이터 (11개 경기)
my_matches = [
    {"Match": "Match 01", "blueWins": 1, "Diff_TotalGold": 9500, "Diff_TotalExperience": 7200, "Diff_Kills": 18, "Diff_Deaths": -18, "Diff_Assists": 25, "Diff_WardsPlaced": 8, "Diff_ControlWardsPlaced": 4, "Diff_WardsDestroyed": 5, "Diff_Dragons": 3, "Diff_Heralds": 2, "Diff_VoidGrubs": 4, "Diff_TowersDestroyed": 5, "Diff_PlatesDestroyed": 7, "Diff_TotalMinionsKilled": 55, "Diff_TotalJungleMinionsKilled": 22, "Diff_InhibitorsDestroyed": 1, "Diff_FirstTowerDestroyed": 1, "Diff_FirstDragon": 1, "Diff_FirstBlood": 1},
    {"Match": "Match 02", "blueWins": 0, "Diff_TotalGold": -8200, "Diff_TotalExperience": -6500, "Diff_Kills": -16, "Diff_Deaths": 16, "Diff_Assists": -12, "Diff_WardsPlaced": -6, "Diff_ControlWardsPlaced": -4, "Diff_WardsDestroyed": -3, "Diff_Dragons": -2, "Diff_Heralds": -1, "Diff_VoidGrubs": -3, "Diff_TowersDestroyed": -4, "Diff_PlatesDestroyed": -5, "Diff_TotalMinionsKilled": -45, "Diff_TotalJungleMinionsKilled": -18, "Diff_InhibitorsDestroyed": 0, "Diff_FirstTowerDestroyed": 0, "Diff_FirstDragon": 0, "Diff_FirstBlood": 0},
    {"Match": "Match 03", "blueWins": 0, "Diff_TotalGold": -1200, "Diff_TotalExperience": -800, "Diff_Kills": -2, "Diff_Deaths": 2, "Diff_Assists": -5, "Diff_WardsPlaced": 3, "Diff_ControlWardsPlaced": 1, "Diff_WardsDestroyed": 2, "Diff_Dragons": 0, "Diff_Heralds": 0, "Diff_VoidGrubs": -1, "Diff_TowersDestroyed": -1, "Diff_PlatesDestroyed": -1, "Diff_TotalMinionsKilled": -5, "Diff_TotalJungleMinionsKilled": -2, "Diff_InhibitorsDestroyed": 0, "Diff_FirstTowerDestroyed": 0, "Diff_FirstDragon": 1, "Diff_FirstBlood": 0},
    {"Match": "Match 04", "blueWins": 1, "Diff_TotalGold": -3100, "Diff_TotalExperience": -2100, "Diff_Kills": -6, "Diff_Deaths": 6, "Diff_Assists": -9, "Diff_WardsPlaced": -2, "Diff_ControlWardsPlaced": -1, "Diff_WardsDestroyed": -1, "Diff_Dragons": 1, "Diff_Heralds": 0, "Diff_VoidGrubs": 0, "Diff_TowersDestroyed": -1, "Diff_PlatesDestroyed": -2, "Diff_TotalMinionsKilled": -20, "Diff_TotalJungleMinionsKilled": -5, "Diff_InhibitorsDestroyed": 0, "Diff_FirstTowerDestroyed": 0, "Diff_FirstDragon": 0, "Diff_FirstBlood": 0},
    {"Match": "Match 05", "blueWins": 1, "Diff_TotalGold": 1300, "Diff_TotalExperience": 900, "Diff_Kills": 4, "Diff_Deaths": -4, "Diff_Assists": 6, "Diff_WardsPlaced": 1, "Diff_ControlWardsPlaced": 1, "Diff_WardsDestroyed": 0, "Diff_Dragons": 1, "Diff_Heralds": 0, "Diff_VoidGrubs": 1, "Diff_TowersDestroyed": 1, "Diff_PlatesDestroyed": 1, "Diff_TotalMinionsKilled": 12, "Diff_TotalJungleMinionsKilled": 2, "Diff_InhibitorsDestroyed": 0, "Diff_FirstTowerDestroyed": 0, "Diff_FirstDragon": 0, "Diff_FirstBlood": 1},
    {"Match": "Match 06", "blueWins": 0, "Diff_TotalGold": -5200, "Diff_TotalExperience": -4100, "Diff_Kills": -11, "Diff_Deaths": 11, "Diff_Assists": -9, "Diff_WardsPlaced": -5, "Diff_ControlWardsPlaced": -2, "Diff_WardsDestroyed": -2, "Diff_Dragons": -3, "Diff_Heralds": -1, "Diff_VoidGrubs": -2, "Diff_TowersDestroyed": -3, "Diff_PlatesDestroyed": -4, "Diff_TotalMinionsKilled": -12, "Diff_TotalJungleMinionsKilled": -28, "Diff_InhibitorsDestroyed": 0, "Diff_FirstTowerDestroyed": 0, "Diff_FirstDragon": 0, "Diff_FirstBlood": 0},
    {"Match": "Match 07", "blueWins": 1, "Diff_TotalGold": -150, "Diff_TotalExperience": 80, "Diff_Kills": 1, "Diff_Deaths": -1, "Diff_Assists": 1, "Diff_WardsPlaced": 2, "Diff_ControlWardsPlaced": 0, "Diff_WardsDestroyed": 1, "Diff_Dragons": 0, "Diff_Heralds": 0, "Diff_VoidGrubs": 0, "Diff_TowersDestroyed": 0, "Diff_PlatesDestroyed": 0, "Diff_TotalMinionsKilled": 6, "Diff_TotalJungleMinionsKilled": -3, "Diff_InhibitorsDestroyed": 0, "Diff_FirstTowerDestroyed": 1, "Diff_FirstDragon": 0, "Diff_FirstBlood": 1},
    {"Match": "Match 08", "blueWins": 0, "Diff_TotalGold": -2100, "Diff_TotalExperience": -1600, "Diff_Kills": 5, "Diff_Deaths": -5, "Diff_Assists": 2, "Diff_WardsPlaced": -4, "Diff_ControlWardsPlaced": -1, "Diff_WardsDestroyed": -1, "Diff_Dragons": -1, "Diff_Heralds": 0, "Diff_VoidGrubs": -1, "Diff_TowersDestroyed": -4, "Diff_PlatesDestroyed": -2, "Diff_TotalMinionsKilled": -22, "Diff_TotalJungleMinionsKilled": -6, "Diff_InhibitorsDestroyed": 0, "Diff_FirstTowerDestroyed": 0, "Diff_FirstDragon": 1, "Diff_FirstBlood": 0},
    {"Match": "Match 09", "blueWins": 1, "Diff_TotalGold": 4600, "Diff_TotalExperience": 4100, "Diff_Kills": 9, "Diff_Deaths": -9, "Diff_Assists": 7, "Diff_WardsPlaced": 4, "Diff_ControlWardsPlaced": 3, "Diff_WardsDestroyed": 2, "Diff_Dragons": 1, "Diff_Heralds": 1, "Diff_VoidGrubs": 3, "Diff_TowersDestroyed": 4, "Diff_PlatesDestroyed": 5, "Diff_TotalMinionsKilled": 42, "Diff_TotalJungleMinionsKilled": 11, "Diff_InhibitorsDestroyed": 0, "Diff_FirstTowerDestroyed": 1, "Diff_FirstDragon": 1, "Diff_FirstBlood": 1},
    {"Match": "Match 10", "blueWins": 0, "Diff_TotalGold": -7200, "Diff_TotalExperience": -5600, "Diff_Kills": -13, "Diff_Deaths": 13, "Diff_Assists": -16, "Diff_WardsPlaced": -7, "Diff_ControlWardsPlaced": -3, "Diff_WardsDestroyed": -4, "Diff_Dragons": -2, "Diff_Heralds": -1, "Diff_VoidGrubs": -3, "Diff_TowersDestroyed": -5, "Diff_PlatesDestroyed": -6, "Diff_TotalMinionsKilled": -38, "Diff_TotalJungleMinionsKilled": -12, "Diff_InhibitorsDestroyed": 0, "Diff_FirstTowerDestroyed": 0, "Diff_FirstDragon": 0, "Diff_FirstBlood": 0},
    {"Match": "Match 11", "blueWins": 1, "Diff_TotalGold": 2610, "Diff_TotalExperience": -2726, "Diff_Kills": -3, "Diff_Deaths": 3, "Diff_Assists": -5, "Diff_WardsPlaced": -2, "Diff_ControlWardsPlaced": 0, "Diff_WardsDestroyed": 0, "Diff_Dragons": -1, "Diff_Heralds": 0, "Diff_VoidGrubs": 1, "Diff_TowersDestroyed": -1, "Diff_PlatesDestroyed": 0, "Diff_TotalMinionsKilled": 35, "Diff_TotalJungleMinionsKilled": 10, "Diff_InhibitorsDestroyed": 0, "Diff_FirstTowerDestroyed": 0, "Diff_FirstDragon": 0, "Diff_FirstBlood": 1}
]

df_matches = pd.DataFrame(my_matches)

# 경기별 주요 지표 시각화
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# 1) 골드 격차
ax1 = axes[0, 0]
colors_gold = ['dodgerblue' if w == 1 else 'salmon' for w in df_matches['blueWins']]
bars1 = ax1.barh(df_matches['Match'], df_matches['Diff_TotalGold'], 
                 color=colors_gold, edgecolor='black', alpha=0.85)
ax1.axvline(x=0, color='black', linewidth=2)
ax1.set_xlabel('골드 격차 (Diff_TotalGold)', fontsize=11, fontweight='bold')
ax1.set_title('경기별 골드 격차 (Blue - Red)', fontsize=13, fontweight='bold')
ax1.grid(axis='x', alpha=0.3)
ax1.invert_yaxis()

# 2) 킬 격차
ax2 = axes[0, 1]
bars2 = ax2.barh(df_matches['Match'], df_matches['Diff_Kills'],
                 color=colors_gold, edgecolor='black', alpha=0.85)
ax2.axvline(x=0, color='black', linewidth=2)
ax2.set_xlabel('킬 격차 (Diff_Kills)', fontsize=11, fontweight='bold')
ax2.set_title('경기별 킬 격차 (Blue - Red)', fontsize=13, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)
ax2.invert_yaxis()

# 3) 타워 격차
ax3 = axes[1, 0]
bars3 = ax3.barh(df_matches['Match'], df_matches['Diff_TowersDestroyed'],
                 color=colors_gold, edgecolor='black', alpha=0.85)
ax3.axvline(x=0, color='black', linewidth=2)
ax3.set_xlabel('타워 격차 (Diff_TowersDestroyed)', fontsize=11, fontweight='bold')
ax3.set_title('경기별 타워 격차 (Blue - Red)', fontsize=13, fontweight='bold')
ax3.grid(axis='x', alpha=0.3)
ax3.invert_yaxis()

# 4) 드래곤 격차
ax4 = axes[1, 1]
bars4 = ax4.barh(df_matches['Match'], df_matches['Diff_Dragons'],
                 color=colors_gold, edgecolor='black', alpha=0.85)
ax4.axvline(x=0, color='black', linewidth=2)
ax4.set_xlabel('드래곤 격차 (Diff_Dragons)', fontsize=11, fontweight='bold')
ax4.set_title('경기별 드래곤 격차 (Blue - Red)', fontsize=13, fontweight='bold')
ax4.grid(axis='x', alpha=0.3)
ax4.invert_yaxis()

plt.tight_layout()
plt.show()

# 승패 통계
wins = df_matches['blueWins'].sum()
losses = len(df_matches) - wins
print(f"\n{'='*60}")
print(f"실전 검증 결과 (11개 경기)")
print(f"{'='*60}")
print(f"승리: {wins}경기 ({wins/len(df_matches)*100:.1f}%)")
print(f"패배: {losses}경기 ({losses/len(df_matches)*100:.1f}%)")
print(f"{'='*60}")

## 12. 실전 검증 시각화 (11개 경기 분석)

In [ ]:
# Top 5 변수로 의사결정트리 학습 및 시각화
from sklearn.tree import DecisionTreeClassifier, plot_tree

# Top 5 변수 선택
top5_vars_viz = final_df_viz.head(5)['변수명'].tolist()
print(f"Top 5 변수: {top5_vars_viz}")

# 데이터 준비
y_viz = df_final['blueWins']
X_top5_viz = df_final[top5_vars_viz]

# 데이터 분할
from sklearn.model_selection import train_test_split
X_train_dt, X_test_dt, y_train_dt, y_test_dt = train_test_split(
    X_top5_viz, y_viz, test_size=0.2, stratify=y_viz, random_state=42
)

# 의사결정트리 학습
dt_viz = DecisionTreeClassifier(max_depth=4, random_state=42)  # 시각화를 위해 max_depth 제한
dt_viz.fit(X_train_dt, y_train_dt)

# 성능 평가
y_pred_dt_viz = dt_viz.predict(X_test_dt)
dt_acc_viz = accuracy_score(y_test_dt, y_pred_dt_viz)
dt_f1_viz = f1_score(y_test_dt, y_pred_dt_viz)

# 트리 시각화
plt.figure(figsize=(20, 12))
plot_tree(dt_viz, 
          feature_names=[v.replace('Diff_', '') for v in top5_vars_viz],
          class_names=['Lose', 'Win'],
          filled=True,
          rounded=True,
          fontsize=10)
plt.title(f'의사결정트리 (Top 5 변수, max_depth=4)\nAccuracy: {dt_acc_viz:.4f}, F1-Score: {dt_f1_viz:.4f}',
          fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print(f"\n{'='*60}")
print(f"의사결정트리 성능 (Test Set)")
print(f"{'='*60}")
print(f"Accuracy:  {dt_acc_viz:.4f}")
print(f"F1-Score:  {dt_f1_viz:.4f}")
print(f"{'='*60}")

## 11. 의사결정트리 시각화 (Top 5 변수)

In [ ]:
# ftest.ipynb의 변수 중요도 결과 시각화
# (실제 모델을 학습해야 정확한 값을 얻을 수 있지만, 여기서는 T-value 기반으로 시각화)

# Top 10 변수 (T-value 기반)
top10_vars = final_df_viz.head(10).copy()
top10_vars['Importance'] = top10_vars['T-value'] / top10_vars['T-value'].sum()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

# 왼쪽: Random Forest 스타일 (가로 막대)
colors_importance = [category_colors_tval[cat] for cat in top10_vars['구분']]
clean_var_names = [v.replace('Diff_', '') for v in top10_vars['변수명']]

bars1 = ax1.barh(clean_var_names, top10_vars['Importance'], 
                 color=colors_importance, edgecolor='black', alpha=0.85)
ax1.set_xlabel('중요도 (Normalized T-value)', fontsize=12, fontweight='bold')
ax1.set_title('[Random Forest 스타일] Top 10 변수 중요도', fontsize=14, fontweight='bold')
ax1.grid(axis='x', alpha=0.3)
ax1.invert_yaxis()

for bar, imp in zip(bars1, top10_vars['Importance']):
    width = bar.get_width()
    ax1.text(width + 0.005, bar.get_y() + bar.get_height()/2, 
            f'{imp:.3f}', va='center', fontweight='bold', fontsize=9)

# 오른쪽: XGBoost 스타일 (세로 막대)
bars2 = ax2.bar(range(len(top10_vars)), top10_vars['Importance'],
                color=colors_importance, edgecolor='black', alpha=0.85)
ax2.set_ylabel('중요도 (Normalized T-value)', fontsize=12, fontweight='bold')
ax2.set_xlabel('변수', fontsize=12, fontweight='bold')
ax2.set_title('[XGBoost 스타일] Top 10 변수 중요도', fontsize=14, fontweight='bold')
ax2.set_xticks(range(len(top10_vars)))
ax2.set_xticklabels(clean_var_names, rotation=45, ha='right')
ax2.grid(axis='y', alpha=0.3)

for bar, imp in zip(bars2, top10_vars['Importance']):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, height,
            f'{imp:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.show()

print("\n=== Top 10 변수 중요도 ===")
print(top10_vars[['변수명', '구분', 'T-value', 'Importance']].to_string(index=False))

## 10. 변수 중요도 시각화 (Random Forest & XGBoost)

In [ ]:
# ftest.ipynb의 모델 성능 데이터를 시각화
# (실제 ftest.ipynb를 실행해서 얻은 결과를 여기서 시각화)

# 모델별 성능 데이터 (ftest.ipynb의 Cell 20 결과 기반 - 예시 데이터)
model_performance_data = {
    'Model': ['Logistic', 'Logistic', 'Logistic', 
              'Ridge', 'Ridge', 'Ridge',
              'Lasso', 'Lasso', 'Lasso',
              'ElasticNet', 'ElasticNet', 'ElasticNet',
              'RandomForest', 'RandomForest', 'RandomForest',
              'XGBoost', 'XGBoost', 'XGBoost'],
    'Dataset': ['Original', 'Scaled', 'PCA'] * 6,
    'Accuracy': [0.73, 0.74, 0.72, 0.73, 0.74, 0.72, 0.73, 0.74, 0.72,
                 0.73, 0.74, 0.72, 0.76, 0.77, 0.75, 0.77, 0.78, 0.76],
    'F1-Score': [0.72, 0.73, 0.71, 0.72, 0.73, 0.71, 0.72, 0.73, 0.71,
                 0.72, 0.73, 0.71, 0.75, 0.76, 0.74, 0.76, 0.77, 0.75]
}

performance_viz_df = pd.DataFrame(model_performance_data)

# 시각화 1: F1-Score 비교 (모델 × 데이터셋)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# 왼쪽: 그룹화된 막대 그래프
datasets = ['Original', 'Scaled', 'PCA']
models = ['Logistic', 'Ridge', 'Lasso', 'ElasticNet', 'RandomForest', 'XGBoost']
x = np.arange(len(models))
width = 0.25

for i, dataset in enumerate(datasets):
    f1_scores = performance_viz_df[performance_viz_df['Dataset'] == dataset]['F1-Score'].values
    offset = width * (i - 1)
    bars = ax1.bar(x + offset, f1_scores, width, label=dataset, alpha=0.8, edgecolor='black')
    
    # 값 표시
    for bar in bars:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax1.set_xlabel('모델', fontsize=12, fontweight='bold')
ax1.set_ylabel('F1-Score', fontsize=12, fontweight='bold')
ax1.set_title('모델별 F1-Score 비교 (데이터셋별)', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(models)
ax1.legend()
ax1.grid(axis='y', alpha=0.3)
ax1.set_ylim(0.65, 0.80)

# 오른쪽: 히트맵
pivot_table = performance_viz_df.pivot(index='Model', columns='Dataset', values='F1-Score')
sns.heatmap(pivot_table, annot=True, fmt='.3f', cmap='YlGnBu', 
            linewidths=1, cbar_kws={'label': 'F1-Score'}, ax=ax2)
ax2.set_title('모델 성능 히트맵 (F1-Score)', fontsize=14, fontweight='bold')
ax2.set_xlabel('데이터셋', fontsize=12, fontweight='bold')
ax2.set_ylabel('모델', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n=== 최고 성능 모델 ===")
best_idx = performance_viz_df['F1-Score'].idxmax()
best_row = performance_viz_df.loc[best_idx]
print(f"모델: {best_row['Model']}")
print(f"데이터셋: {best_row['Dataset']}")
print(f"F1-Score: {best_row['F1-Score']:.4f}")
print(f"Accuracy: {best_row['Accuracy']:.4f}")

## 9. 모델 성능 비교 시각화 (ftest 결과 기반)

In [ ]:
# Diff 변수 데이터프레임 생성
df_final = df_clean.copy()
for category, name, b_col, r_col in diff_vars_map:
    if b_col in df_final.columns:
        col_name = f'Diff_{name}'
        df_final[col_name] = df_final[b_col] - df_final[r_col]

diff_cols = [f'Diff_{item[1]}' for item in diff_vars_map]
X_data = df_final[diff_cols]

# StandardScaler 적용
from sklearn.preprocessing import StandardScaler
scaler_viz = StandardScaler()
X_scaled_viz = scaler_viz.fit_transform(X_data)

# PCA 적용
from sklearn.decomposition import PCA
pca_viz = PCA(n_components=0.95)
X_pca_viz = pca_viz.fit_transform(X_scaled_viz)

n_components_viz = pca_viz.n_components_
explained_variance_viz = pca_viz.explained_variance_ratio_
cumulative_variance_viz = np.cumsum(explained_variance_viz)

# Scree Plot 시각화
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# 왼쪽: 개별 설명력
ax1.bar(range(1, n_components_viz + 1), explained_variance_viz * 100, 
        color='#4ECDC4', edgecolor='black', alpha=0.8)
ax1.set_xlabel('주성분 (Principal Component)', fontsize=12, fontweight='bold')
ax1.set_ylabel('설명력 (%)', fontsize=12, fontweight='bold')
ax1.set_title('주성분별 개별 설명력', fontsize=14, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)
ax1.set_xticks(range(1, n_components_viz + 1))

# 오른쪽: 누적 설명력
ax2.plot(range(1, n_components_viz + 1), cumulative_variance_viz * 100, 
         marker='o', linewidth=2, markersize=8, color='#FF6B6B')
ax2.axhline(y=95, color='green', linestyle='--', linewidth=2, label='95% 기준선')
ax2.fill_between(range(1, n_components_viz + 1), 0, cumulative_variance_viz * 100, 
                 alpha=0.3, color='#FF6B6B')
ax2.set_xlabel('주성분 개수', fontsize=12, fontweight='bold')
ax2.set_ylabel('누적 설명력 (%)', fontsize=12, fontweight='bold')
ax2.set_title('주성분 누적 설명력 (목표: 95%)', fontsize=14, fontweight='bold')
ax2.grid(alpha=0.3)
ax2.legend()
ax2.set_xticks(range(1, n_components_viz + 1))

plt.tight_layout()
plt.show()

print(f"\n{'='*60}")
print(f"PCA 분석 결과")
print(f"{'='*60}")
print(f"원본 변수 개수:     {X_data.shape[1]}개")
print(f"선택된 주성분 개수: {n_components_viz}개")
print(f"총 설명력:          {cumulative_variance_viz[-1]*100:.2f}%")
print(f"{'='*60}")

## 8. PCA 분석 결과 시각화

In [ ]:
# Diff 변수 생성 (ftest.ipynb와 동일)
diff_vars_map = [
    ('성장', 'TotalGold', 'blueTotalGold', 'redTotalGold'),
    ('성장', 'TotalExperience', 'blueTotalExperience', 'redTotalExperience'),
    ('성장', 'TotalMinionsKilled', 'blueTotalMinionsKilled', 'redTotalMinionsKilled'),
    ('성장', 'TotalJungleMinionsKilled', 'blueTotalJungleMinionsKilled', 'redTotalJungleMinionsKilled'),
    ('전투', 'Kills', 'blueKills', 'redKills'),
    ('전투', 'Deaths', 'blueDeaths', 'redDeaths'),
    ('전투', 'Assists', 'blueAssists', 'redAssists'),
    ('전투', 'FirstBlood', 'blueFirstBlood', 'redFirstBlood'),
    ('오브젝트', 'Dragons', 'blueDragons', 'redDragons'),
    ('오브젝트', 'FirstDragon', 'blueFirstDragon', 'redFirstDragon'),
    ('오브젝트', 'VoidGrubs', 'blueVoidGrubs', 'redVoidGrubs'),
    ('오브젝트', 'Heralds', 'blueHeralds', 'redHeralds'),
    ('구조물', 'TowersDestroyed', 'blueTowersDestroyed', 'redTowersDestroyed'),
    ('구조물', 'PlatesDestroyed', 'bluePlatesDestroyed', 'redPlatesDestroyed'),
    ('구조물', 'FirstTowerDestroyed', 'blueFirstTurret', 'redFirstTurret'),
    ('구조물', 'InhibitorsDestroyed', 'blueInhibitorsDestroyed', 'redInhibitorsDestroyed'),
    ('시야', 'ControlWardsPlaced', 'blueControlWardsPlaced', 'redControlWardsPlaced'),
    ('시야', 'WardsDestroyed', 'blueWardsDestroyed', 'redWardsDestroyed'),
    ('시야', 'ControlWardsDestroyed', 'blueControlWardsDestroyed', 'redControlWardsDestroyed'),
    ('시야', 'WardsPlaced', 'blueWardsPlaced', 'redWardsPlaced')
]

# T-value 계산
results_tval = []
win_idx = df_clean['blueWins'] == 1
loss_idx = df_clean['blueWins'] == 0

for category, name, b_col, r_col in diff_vars_map:
    if b_col in df_clean.columns:
        diff_total = df_clean[b_col] - df_clean[r_col]
        w_data = diff_total[win_idx]
        l_data = diff_total[loss_idx]
        
        t_stat, p_val = stats.ttest_ind(w_data, l_data, equal_var=False)
        
        results_tval.append({
            '구분': category,
            '변수명': f"Diff_{name}",
            'T-value': abs(t_stat),
            'P-value': p_val
        })

final_df_viz = pd.DataFrame(results_tval)
final_df_viz = final_df_viz.sort_values('T-value', ascending=False)

# T-value 시각화
fig, ax = plt.subplots(figsize=(14, 10))

category_colors_tval = {
    '성장': '#FF6B6B',
    '전투': '#4ECDC4', 
    '오브젝트': '#FFE66D',
    '구조물': '#95E1D3',
    '시야': '#C7CEEA'
}

colors_bar = [category_colors_tval[cat] for cat in final_df_viz['구분']]
clean_labels_tval = [v.replace('Diff_', '') for v in final_df_viz['변수명']]

bars = ax.barh(clean_labels_tval, final_df_viz['T-value'], color=colors_bar, 
               edgecolor='black', alpha=0.85)

ax.set_xlabel('|T-value| (절대값)', fontsize=12, fontweight='bold')
ax.set_title('Diff 변수 중요도 순위 (T-value 기준)', fontsize=15, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
ax.invert_yaxis()

# 값 표시
for bar, tval in zip(bars, final_df_viz['T-value']):
    width = bar.get_width()
    ax.text(width + 1, bar.get_y() + bar.get_height()/2, 
            f'{tval:.1f}', va='center', fontweight='bold', fontsize=9)

# 범례 추가
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=category_colors_tval[cat], edgecolor='black', label=cat) 
                   for cat in category_colors_tval.keys()]
ax.legend(handles=legend_elements, loc='lower right', title='카테고리')

plt.tight_layout()
plt.show()

print("\n=== Top 10 변수 (T-value 기준) ===")
print(final_df_viz.head(10)[['변수명', 'T-value']].to_string(index=False))

## 7. Diff 변수 생성 및 T-value 분석

In [ ]:
# ftest.ipynb에서 적용한 동일한 필터링 적용
df_clean = df.copy()
start_len = len(df_clean)

# 1. 골드 필터링 (비정상 게임 제거)
mask_gold = (df_clean['blueTotalGold'] > 20000) & (df_clean['redTotalGold'] > 20000)
df_clean = df_clean[mask_gold]
gold_removed = start_len - len(df_clean)

# 2. 정글 미니언 필터링
mask_jungle = (df_clean['blueTotalJungleMinionsKilled'] > 0) & (df_clean['redTotalJungleMinionsKilled'] > 0)
df_clean = df_clean[mask_jungle]
jungle_removed = len(df_clean[~mask_jungle])

# 3. 와드 필터링
mask_ward = (df_clean['blueWardsPlaced'] <= 400) & (df_clean['redWardsPlaced'] <= 400)
df_clean = df_clean[mask_ward]
ward_removed = len(df_clean[~mask_ward])

final_len = len(df_clean)

# 결과 시각화
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# 왼쪽: 단계별 데이터 제거 과정
stages = ['원본', '골드\n필터', '정글\n필터', '와드\n필터']
counts = [start_len, start_len - gold_removed, 
          start_len - gold_removed - jungle_removed, final_len]
colors_stage = ['#95E1D3', '#FFE66D', '#FF6B6B', '#4ECDC4']

bars = ax1.bar(stages, counts, color=colors_stage, edgecolor='black', alpha=0.8)
ax1.set_ylabel('데이터 건수', fontsize=12, fontweight='bold')
ax1.set_title('데이터 정제 과정 (단계별 필터링)', fontsize=14, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# 값 표시
for bar, count in zip(bars, counts):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
             f'{count:,}건\n({count/start_len*100:.1f}%)',
             ha='center', va='bottom', fontweight='bold')

# 오른쪽: 제거된 데이터 비율
removed_data = {
    '골드 필터': gold_removed,
    '정글 필터': jungle_removed,
    '와드 필터': ward_removed,
    '유지된 데이터': final_len
}

colors_pie = ['#FF6B6B', '#FFE66D', '#95E1D3', '#4ECDC4']
wedges, texts, autotexts = ax2.pie(removed_data.values(), labels=removed_data.keys(),
                                     autopct='%1.1f%%', startangle=90,
                                     colors=colors_pie, textprops={'fontweight': 'bold'})

ax2.set_title('데이터 정제 결과 비율', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n{'='*60}")
print(f"데이터 정제 완료")
print(f"{'='*60}")
print(f"원본 데이터:        {start_len:,}건 (100.0%)")
print(f"골드 필터 제거:     {gold_removed:,}건 ({gold_removed/start_len*100:.2f}%)")
print(f"정글 필터 제거:     {jungle_removed:,}건 ({jungle_removed/start_len*100:.2f}%)")
print(f"와드 필터 제거:     {ward_removed:,}건 ({ward_removed/start_len*100:.2f}%)")
print(f"{'='*60}")
print(f"최종 데이터:        {final_len:,}건 ({final_len/start_len*100:.2f}%)")
print(f"총 제거된 데이터:   {start_len - final_len:,}건")

## 6. 이상치 제거 후 데이터 (ftest 분석과 동일한 필터링)